# 03 - SGD Sampling and Escape Times

This notebook studies ensemble behavior of SGD and escape times from local minima, connecting to Kramers formula from statistical physics.

**Converted from:** `sgd_example(2)_sampling.nb` and `SDE_escapetime.nb` (Mathematica)

## Contents:
1. Ensemble trajectory generation
2. Escape time calculations
3. Kramers formula verification
4. Learning rate dependence
5. Barrier crossing analysis

## Background

When SGD is trapped in a local minimum, it must rely on stochastic noise to escape. The mean escape time follows Kramers formula:

$$\tau_{\text{escape}} \sim \frac{1}{D} e^{\Delta E / D}$$

where $\Delta E$ is the barrier height and $D$ is the diffusion coefficient (related to learning rate).

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
from tqdm import tqdm

sys.path.insert(0, str(Path.cwd() / 'utils'))

from sgd_simulator import SGDSimulator, SGDConfig
from loss_functions import (
    smooth_nonlinear_loss,
    smooth_nonlinear_gradient,
    generate_noisy_data,
    SmoothNonlinearLoss
)
from sde_tools import (
    compute_escape_time,
    mean_escape_time_kramers,
    sgd_as_langevin,
    estimate_gradient_variance,
    langevin_dynamics,
    drift_from_loss
)
from visualization import (
    plot_loss_landscape,
    plot_escape_times,
    plot_trajectories
)

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✓ Libraries imported successfully!")

## 1. Setup: Create Loss Landscape with Barrier

We'll use our standard nonlinear loss function which has multiple local minima separated by barriers.

In [ ]:
# Generate data
np.random.seed(42)

x_data, y_data = generate_noisy_data(
    x_range=(-3, 3),
    n_points=20,
    n_samples_per_point=5,
    noise_std=0.5,
    p=1.0,
    random_state=42
)

loss_obj = SmoothNonlinearLoss(p=1.0)

# Visualize the loss landscape
fig, ax = plt.subplots(figsize=(12, 10))
param_range = ((-1, 3), (-1, 3))

plot_loss_landscape(
    loss_fn=loss_obj,
    x_data=x_data,
    y_data=y_data,
    param_range=param_range,
    n_points=100,
    contour_levels=40,
    ax=ax,
    title='Loss Landscape with Multiple Minima'
)

plt.tight_layout()
plt.show()

print("Loss landscape created with multiple local minima")

## 2. Generate Ensemble of SGD Trajectories

Let's run many SGD trajectories from the same starting point to study statistical properties.

In [ ]:
# Configuration for ensemble
learning_rate = 0.03
batch_size = 5
n_iterations = 3000
n_trajectories = 20

# Starting point (in a local minimum region)
initial_params = np.array([0.3, 0.3])

# Run ensemble
sgd_config = SGDConfig(
    learning_rate=learning_rate,
    batch_size=batch_size,
    n_iterations=n_iterations,
    random_state=None  # Different random seed for each
)

print(f"Generating {n_trajectories} SGD trajectories...")
trajectories = []
final_positions = []

for i in tqdm(range(n_trajectories)):
    simulator = SGDSimulator(sgd_config)
    traj, iters = simulator.run_trajectory(
        initial_params=initial_params,
        gradient_fn=loss_obj.gradient,
        x_data=x_data,
        y_data=y_data,
        save_every=10
    )
    trajectories.append(traj)
    final_positions.append(traj[-1])

final_positions = np.array(final_positions)
print(f"✓ Generated {n_trajectories} trajectories")
print(f"Final position mean: {final_positions.mean(axis=0)}")
print(f"Final position std: {final_positions.std(axis=0)}")

In [ ]:
# Plot ensemble of trajectories
fig, ax = plt.subplots(figsize=(14, 11))

# Plot loss landscape
(a_min, a_max), (b_min, b_max) = param_range
n_points = 100
a_vals = np.linspace(a_min, a_max, n_points)
b_vals = np.linspace(b_min, b_max, n_points)
A, B = np.meshgrid(a_vals, b_vals)

Z = np.zeros_like(A)
for i in range(n_points):
    for j in range(n_points):
        params = np.array([A[i, j], B[i, j]])
        Z[i, j] = loss_obj(params, x_data, y_data)

contourf = ax.contourf(A, B, Z, levels=30, cmap='viridis', alpha=0.3)
ax.contour(A, B, Z, levels=30, cmap='viridis', alpha=0.6)
plt.colorbar(contourf, ax=ax, label='Loss')

# Plot all trajectories
for i, traj in enumerate(trajectories):
    ax.plot(traj[:, 0], traj[:, 1], '-', linewidth=1.5, alpha=0.6, color='red')
    if i == 0:
        ax.plot(traj[0, 0], traj[0, 1], 'go', markersize=12, 
               label='Start', markeredgecolor='black', markeredgewidth=2)

# Plot final positions
ax.scatter(final_positions[:, 0], final_positions[:, 1], 
          s=100, c='yellow', marker='*', edgecolors='black',
          linewidths=2, label='Final positions', zorder=10)

ax.set_xlabel('Parameter a', fontsize=12)
ax.set_ylabel('Parameter b', fontsize=12)
ax.set_title(f'Ensemble of {n_trajectories} SGD Trajectories', fontsize=14)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Different trajectories explore different parts of the landscape!")

## 3. Escape Time Analysis

Now let's compute escape times: how long it takes SGD to leave a region around the initial point.

We'll define a "target region" (escape boundary) and measure when trajectories cross it.

In [ ]:
# Define escape region (circle around starting point)
escape_radius = 0.5

def is_escaped(params):
    """Check if parameters have escaped from initial region."""
    distance = np.linalg.norm(params - initial_params)
    return distance > escape_radius

# Compute escape times for our trajectories
escape_times = []
escaped_count = 0

for traj in trajectories:
    for t, params in enumerate(traj):
        if is_escaped(params):
            escape_times.append(t * 10)  # Convert to iterations
            escaped_count += 1
            break
    else:
        # Didn't escape
        escape_times.append(n_iterations)

escape_times = np.array(escape_times)

print(f"Escaped: {escaped_count}/{n_trajectories} trajectories")
print(f"Mean escape time: {escape_times.mean():.1f} iterations")
print(f"Median escape time: {np.median(escape_times):.1f} iterations")
print(f"Std escape time: {escape_times.std():.1f} iterations")

In [ ]:
# Plot histogram of escape times
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(escape_times, bins=20, density=True, alpha=0.7, 
       color='steelblue', edgecolor='black')
ax.axvline(escape_times.mean(), color='red', linestyle='--', 
          linewidth=2, label=f'Mean = {escape_times.mean():.0f}')
ax.axvline(np.median(escape_times), color='orange', linestyle='--',
          linewidth=2, label=f'Median = {np.median(escape_times):.0f}')

ax.set_xlabel('Escape Time (iterations)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Distribution of Escape Times', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Kramers Formula Verification

Kramers formula predicts that escape time depends exponentially on the barrier height and inversely on the diffusion coefficient:

$$\tau_{\text{escape}} \propto \frac{1}{\omega} e^{\Delta E / D}$$

where $D \propto \eta^2$ (learning rate squared).

Let's verify this by measuring escape times for different learning rates.

In [ ]:
# Test different learning rates
learning_rates = np.logspace(-2.5, -0.5, 8)  # From 0.003 to 0.3
n_traj_per_lr = 30
batch_size = 5

mean_escape_times = []
std_escape_times = []

print("Computing escape times for different learning rates...")
for lr in tqdm(learning_rates):
    escape_times_lr = []
    
    sgd_config = SGDConfig(
        learning_rate=lr,
        batch_size=batch_size,
        n_iterations=5000,
        random_state=None
    )
    
    for _ in range(n_traj_per_lr):
        simulator = SGDSimulator(sgd_config)
        traj, iters = simulator.run_trajectory(
            initial_params=initial_params,
            gradient_fn=loss_obj.gradient,
            x_data=x_data,
            y_data=y_data,
            save_every=5
        )
        
        # Find escape time
        for t, params in enumerate(traj):
            if is_escaped(params):
                escape_times_lr.append(t * 5)
                break
        else:
            escape_times_lr.append(5000)
    
    mean_escape_times.append(np.mean(escape_times_lr))
    std_escape_times.append(np.std(escape_times_lr))

mean_escape_times = np.array(mean_escape_times)
std_escape_times = np.array(std_escape_times)

print("✓ Completed escape time analysis")

In [ ]:
# Compute theoretical prediction from Kramers formula
# Estimate barrier height and diffusion
barrier_height = 2.0  # Approximate from loss landscape
grad_var = estimate_gradient_variance(
    loss_obj.gradient, initial_params, x_data, y_data, n_samples=100
)

theoretical_escape_times = []
for lr in learning_rates:
    D = sgd_as_langevin(lr, batch_size, len(x_data), grad_var)
    # Kramers: tau ~ exp(barrier/D) / omega
    # Simplified version (omega ~ 1)
    if D > 0:
        tau = np.exp(barrier_height / D) / 100  # Scale factor
    else:
        tau = np.inf
    theoretical_escape_times.append(tau)

theoretical_escape_times = np.array(theoretical_escape_times)

# Rescale theoretical curve to match scale
scale_factor = mean_escape_times[3] / theoretical_escape_times[3]
theoretical_escape_times *= scale_factor

print(f"Barrier height estimate: {barrier_height}")
print(f"Gradient variance: {grad_var:.4f}")

In [ ]:
# Plot escape time vs learning rate
fig, ax = plt.subplots(figsize=(12, 8))

# Plot empirical data with error bars
ax.errorbar(learning_rates, mean_escape_times, yerr=std_escape_times,
           fmt='o-', markersize=8, linewidth=2, capsize=5,
           label='Empirical (SGD)', color='blue', alpha=0.8)

# Plot theoretical curve
ax.plot(learning_rates, theoretical_escape_times, '--',
       linewidth=2.5, label='Theoretical (Kramers)', color='red')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Learning Rate η', fontsize=13)
ax.set_ylabel('Mean Escape Time (iterations)', fontsize=13)
ax.set_title('Escape Time vs Learning Rate\n(Kramers Formula Verification)', 
            fontsize=14)
ax.grid(True, alpha=0.3, which='both')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print("Escape time decreases with learning rate, as predicted by Kramers formula!")

## 5. Visualize Escape Dynamics

Let's visualize how trajectories escape for different learning rates.

In [ ]:
# Generate example trajectories for low, medium, and high learning rates
lr_examples = [0.01, 0.05, 0.15]
colors_lr = ['blue', 'green', 'red']
labels_lr = ['Low η=0.01', 'Medium η=0.05', 'High η=0.15']

fig, ax = plt.subplots(figsize=(14, 11))

# Plot loss landscape
contourf = ax.contourf(A, B, Z, levels=30, cmap='gray', alpha=0.2)
ax.contour(A, B, Z, levels=30, cmap='gray', alpha=0.4)

# Draw escape boundary
circle = plt.Circle(initial_params, escape_radius, color='black', 
                   fill=False, linestyle='--', linewidth=3, label='Escape boundary')
ax.add_patch(circle)

# Generate and plot trajectories
for lr, color, label in zip(lr_examples, colors_lr, labels_lr):
    sgd_config = SGDConfig(learning_rate=lr, batch_size=5, 
                          n_iterations=1000, random_state=42)
    simulator = SGDSimulator(sgd_config)
    traj, _ = simulator.run_trajectory(
        initial_params, loss_obj.gradient, x_data, y_data, save_every=5
    )
    
    ax.plot(traj[:, 0], traj[:, 1], '-', linewidth=2, 
           color=color, alpha=0.8, label=label)

ax.plot(initial_params[0], initial_params[1], 'ko', markersize=15,
       markeredgewidth=2, label='Start')

ax.set_xlabel('Parameter a', fontsize=12)
ax.set_ylabel('Parameter b', fontsize=12)
ax.set_title('Escape Dynamics for Different Learning Rates', fontsize=14)
ax.set_xlim(param_range[0])
ax.set_ylim(param_range[1])
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Higher learning rates lead to faster, more exploratory escapes!")

## Summary

In this notebook, we explored:

1. **Ensemble Behavior**: Multiple SGD runs show statistical variation
2. **Escape Times**: Measured how long it takes to leave a local region
3. **Kramers Formula**: Verified exponential dependence on barrier height and diffusion
4. **Learning Rate Effects**: Higher learning rates → faster escape, more exploration
5. **Barrier Crossing**: Visualized how stochastic noise enables escape from local minima

**Key Findings:**
- Escape time: $\tau \sim e^{\Delta E / D}$ where $D \propto \eta^2$
- Small learning rates: trapped in local minima for long times
- Large learning rates: fast exploration but poor convergence
- This explains the bias-variance tradeoff in learning rate selection

**Practical Implications:**
- Learning rate warmup helps escape bad initializations
- Learning rate schedules balance exploration and convergence
- Understanding escape times informs hyperparameter selection

**Next Steps:**
- Notebook 4 will explore smooth approximations and alternative loss functions
- Notebook 5 will dive deeper into 2D SDE escape time analysis